# 01 — Baseline VLM

**EFREI 2025-2026 · Solution Delivery · Filière Data**

> **Position non clinique.** Ce dépôt n'est pas un dispositif médical. Les sorties ne constituent pas un avis médical.

## Objectif

1. Charger le dataset synthétique (30 cas)
2. Lancer l'inférence **toy baseline** sur tous les cas (reproductible sans API)
3. Calculer les métriques : accuracy, macro F1, warning_rate, json_valid_rate
4. Sauvegarder les prédictions et métriques dans `eval/outputs/`
5. *(Optionnel)* Lancer l'inférence **VLM Claude** si `ANTHROPIC_API_KEY` est définie

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
print('ROOT:', ROOT)

## 1 — Chargement du dataset

In [ ]:
import csv
import json
import pandas as pd

cases_path = ROOT / 'data' / 'synthetic_cases.csv'
with cases_path.open(newline='', encoding='utf-8') as f:
    cases = list(csv.DictReader(f))

df = pd.DataFrame(cases)
print(f'{len(df)} cas chargés')
display(df[['case_id', 'label', 'quality', 'split']].head(10))
print()
display(df['label'].value_counts())

## 2 — Inférence toy baseline

Le classificateur **toy** est déterministe et ne nécessite pas de clé API.
Il prédit à partir du nom de fichier — c'est un outil de validation du pipeline, pas un modèle médical.

In [ ]:
from src.inference import toy_predict
from src.guardrails import apply_safety_guardrails, WARNING_TEXT

rows = []
for case in cases:
    image_path = ROOT / case['image_path']
    pred = apply_safety_guardrails(toy_predict(image_path, mode='baseline'))

    rows.append({
        'case_id':        case['case_id'],
        'label':          case['label'],
        'predicted_class': pred['predicted_class'],
        'confidence':      pred['confidence'],
        'image_quality':   pred['image_quality'],
        'json_valid':      True,
        'warning':         pred.get('warning', ''),
        'model_name':      pred.get('model_name', ''),
        'latency_ms':      pred.get('latency_ms', 0),
    })

df_pred = pd.DataFrame(rows)
display(df_pred)

## 3 — Métriques

In [ ]:
from src.metrics import accuracy, macro_f1, confusion_counts, summarize_metrics

metrics = summarize_metrics(rows)

print('=== Métriques baseline ===')
print(f"Accuracy      : {metrics['accuracy']:.1%}")
print(f"Macro F1      : {metrics['macro_f1']:.1%}")
print(f"Warning rate  : {metrics['warning_rate']:.1%}")
print(f"JSON valid    : {metrics['json_valid_rate']:.1%}")
print(f"Uncertain rate: {metrics['uncertain_rate']:.1%}")
print()

cm = confusion_counts(
    df_pred['label'].tolist(),
    df_pred['predicted_class'].tolist()
)
print('Matrice de confusion (true__pred):')
for k, v in sorted(cm.items()):
    print(f'  {k}: {v}')

## 4 — Sauvegarde des résultats

In [ ]:
out_dir = ROOT / 'eval' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

df_pred.to_csv(out_dir / 'baseline_predictions.csv', index=False)
(out_dir / 'baseline_metrics.json').write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False),
    encoding='utf-8'
)

print('Fichiers sauvegardés dans', out_dir)
print('  baseline_predictions.csv')
print('  baseline_metrics.json')

## 5 — (Optionnel) Inférence VLM Claude

Nécessite `ANTHROPIC_API_KEY`. Si la clé est absente, la cellule l'indique et s'arrête proprement.

Le modèle utilisé en mode baseline est `claude-haiku-4-5-20251001` — rapide et économique.

In [ ]:
import os

if not os.environ.get('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY non définie — mode toy uniquement.')
    print('Pour activer le VLM :')
    print('  Windows : $env:ANTHROPIC_API_KEY = "sk-ant-..."')
    print('  Linux/Mac : export ANTHROPIC_API_KEY=sk-ant-...')
else:
    from src.inference import vlm_predict

    vlm_rows = []
    # Limité à 5 cas pour maîtriser les coûts API
    for case in cases[:5]:
        image_path = ROOT / case['image_path']
        pred = apply_safety_guardrails(vlm_predict(image_path, mode='baseline'))
        vlm_rows.append({
            'case_id':         case['case_id'],
            'ground_truth':    case['label'],
            'predicted_class': pred['predicted_class'],
            'confidence':      pred['confidence'],
            'model_name':      pred.get('model_name', ''),
            'latency_ms':      pred.get('latency_ms', 0),
        })
        print(f"[{case['case_id']}] GT={case['label']:22s} PRED={pred['predicted_class']:22s} CONF={pred['confidence']:.2f}")

    print()
    display(pd.DataFrame(vlm_rows))

## Rappels de position non clinique

- Ce notebook est **pédagogique**. Les prédictions ne sont pas validées médicalement.
- Les images sont synthétiques. Les labels sont définis par construction, pas par un radiologue.
- Ne jamais utiliser ce système pour diagnostiquer, trier ou orienter un patient réel.

> *Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise.*